# 05 — Frequency Multiplexing

**Engineering question**

How can one integrated photonic device provide many quantum channels simultaneously?

Notebook 01 showed why hardware replication becomes a scaling constraint.  
Notebook 05 introduces the replacement strategy:

```text
replicate fewer devices
→ exploit many optical frequencies
→ treat modes as channels
→ prepare for quantum frequency combs
```

Frequency multiplexing changes the scaling variable from **number of devices** to **number of accessible optical modes**.


## Setup

This notebook creates:

```text
figures/
results/csv/
results/json/
results/05_outputs.zip
```

The final cell supports both:

- **Google Colab**: starts a real browser download with `files.download(...)`
- **Jupyter**: displays a clickable `FileLink`


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. From replication to modal scaling

Replication assigns a channel to each physical device.

Frequency multiplexing assigns many channels to many optical frequencies inside one device.

The physical hardware is not gone.  
The scaling resource has changed.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

# Left: many replicated devices
left_x = [0.8, 0.8, 0.8, 0.8]
left_y = [3.5, 2.6, 1.7, 0.8]
for i, (x, y) in enumerate(zip(left_x, left_y), 1):
    rect = plt.Rectangle((x - 0.35, y - 0.22), 0.7, 0.44, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, f"Device {i}", ha="center", va="center", fontsize=10)

ax.text(0.8, 4.15, "Replication", ha="center", fontsize=14, weight="bold")
ax.text(0.8, 0.15, "one device → one channel", ha="center", fontsize=11)

# Right: one resonator with modes
circle = plt.Circle((3.7, 2.15), 0.75, fill=False, linewidth=3)
ax.add_patch(circle)
ax.text(3.7, 2.15, "Integrated\nresonator", ha="center", va="center", fontsize=12)

mode_x = np.linspace(5.3, 8.7, 8)
for i, mx in enumerate(mode_x, 1):
    ax.plot([mx, mx], [1.1, 3.2], linewidth=2)
    ax.scatter([mx], [3.2], s=80)
    ax.text(mx, 0.82, f"ω{i}", ha="center", fontsize=10)

ax.text(7.0, 4.15, "Frequency multiplexing", ha="center", fontsize=14, weight="bold")
ax.text(7.0, 0.15, "one device → many optical modes", ha="center", fontsize=11)

# Arrow between concepts
ax.annotate("", xy=(2.75, 2.15), xytext=(1.55, 2.15), arrowprops=dict(arrowstyle="->", linewidth=2.5))
ax.text(2.15, 2.38, "shift scaling\nresource", ha="center", fontsize=10)

ax.set_title("From Replicated Devices to Modal Scaling", fontsize=16)
ax.set_xlim(0, 9.4)
ax.set_ylim(-0.1, 4.6)
ax.axis("off")

savefig(fig, "05_replication_to_modes.png")
plt.show()


## 2. Frequency comb as a mode ladder

A resonator supports discrete optical modes.

A frequency comb is a regularly spaced set of modes:

```text
ωₙ = ω₀ + nΔω
```

The spacing is the free spectral range, often written as FSR.


In [ ]:
n = np.arange(-10, 11)
freq = n
amplitude = np.ones_like(n, dtype=float)

fig, ax = plt.subplots(figsize=(10, 3.8))

for ni, amp in zip(freq, amplitude):
    ax.plot([ni, ni], [0, amp], linewidth=2)
    ax.scatter([ni], [amp], s=65)

ax.axvline(0, linestyle="--", linewidth=1.5, alpha=0.7)
ax.text(0, 1.18, "pump / center mode\nω₀", ha="center", fontsize=10)

# mark spacing
ax.annotate("", xy=(1, 0.35), xytext=(0, 0.35), arrowprops=dict(arrowstyle="<->", linewidth=1.5))
ax.text(0.5, 0.42, "Δω / FSR", ha="center", fontsize=10)

ax.set_title("Frequency Comb: Equally Spaced Optical Modes", fontsize=16)
ax.set_xlabel("Mode index n")
ax.set_ylabel("Resonance")
ax.set_yticks([])
ax.set_xlim(-10.8, 10.8)
ax.set_ylim(0, 1.35)
ax.grid(True, axis="x", alpha=0.15)

savefig(fig, "05_frequency_comb.png")
plt.show()


## 3. Channel scaling through modes

This figure extends the Notebook 01 comparison.

The point is not that modal scaling is free.  
The point is that channel count can grow inside a shared physical platform.


In [ ]:
effort = np.arange(1, 21)

replication_channels = effort
multiplexed_channels = np.round(1.2 * effort**1.8).astype(int)

fig, ax = plt.subplots(figsize=(8.5, 5))

ax.plot(effort, replication_channels, "o-", linewidth=2.6, label="Replication: channels scale with devices")
ax.plot(effort, multiplexed_channels, "s-", linewidth=2.6, label="Multiplexing: channels scale with modes")

ax.set_title("Channel Scaling: Devices versus Frequency Modes", fontsize=16)
ax.set_xlabel("Engineering effort / design generation")
ax.set_ylabel("Available optical channels")
ax.grid(True, alpha=0.3)
ax.legend()

savefig(fig, "05_channel_scaling.png")
plt.show()


## 4. Multiplexing pipeline

The basic engineering pipeline is:

```text
laser
→ microresonator
→ frequency comb
→ many optical modes
→ quantum channels
```

Later notebooks refine this:

- Notebook 07: quantum frequency combs
- Notebook 11: Kerr nonlinear enhancement
- Notebook 13: two-mode squeezing


In [ ]:
labels = [
    "Laser",
    "Microresonator",
    "Frequency\ncomb",
    "Many optical\nmodes",
    "Quantum\nchannels",
]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(11, 3.2))

for xi, label in zip(x, labels):
    rect = plt.Rectangle((xi - 0.38, -0.25), 0.76, 0.5, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(xi, 0, label, ha="center", va="center", fontsize=11)

for i in range(len(labels) - 1):
    ax.annotate("", xy=(i + 0.58, 0), xytext=(i + 0.4, 0), arrowprops=dict(arrowstyle="->", linewidth=2))

ax.set_title("Frequency Multiplexing Pipeline", fontsize=16)
ax.set_xlim(-0.7, len(labels) - 0.3)
ax.set_ylim(-0.7, 0.7)
ax.axis("off")

savefig(fig, "05_multiplexing_pipeline.png")
plt.show()


## 5. Modes as resources

Once frequencies are addressable, modes can be treated as resources.

At this stage they are not yet squeezed or entangled.

The notebook only establishes the resource axis:

```text
frequency modes can carry distinguishable channels
```


In [ ]:
modes = np.arange(-6, 7)

fig, ax = plt.subplots(figsize=(10, 3.8))

for m in modes:
    ax.plot([m, m], [0, 1.0], linewidth=2)
    ax.scatter([m], [1.0], s=75)

ax.text(0, 1.32, "one integrated resonator", ha="center", fontsize=13, weight="bold")
ax.text(0, -0.32, "frequency modes become addressable channel resources", ha="center", fontsize=12)

for m in [-6, -3, 0, 3, 6]:
    label = "ω₀" if m == 0 else f"ω₀{m:+d}Δω"
    ax.text(m, -0.08, label, ha="center", va="top", fontsize=9)

ax.set_title("Frequency Modes as Channel Resources", fontsize=16)
ax.set_xlim(-6.8, 6.8)
ax.set_ylim(-0.55, 1.55)
ax.axis("off")

savefig(fig, "05_modes_as_resources.png")
plt.show()


## 6. Engineering summary

Frequency multiplexing changes the scaling burden.

It does not remove every constraint.

It reduces the need to duplicate every physical subsystem for every channel.


In [ ]:
rows = [
    ("Lasers / sources", "many copies", "shared pump / fewer sources"),
    ("Photonic chips", "many devices", "one integrated resonator"),
    ("Channels", "hardware-defined", "frequency-defined"),
    ("Alignment", "many alignments", "shared platform"),
    ("Scaling variable", "device count", "mode count"),
]

fig, ax = plt.subplots(figsize=(10, 4.4))
ax.axis("off")

table = ax.table(
    cellText=[[a, b, c] for a, b, c in rows],
    colLabels=["Resource", "Replication", "Frequency multiplexing"],
    loc="center",
    cellLoc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(10.5)
table.scale(1, 1.7)

for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")

ax.set_title("Engineering Summary: What Multiplexing Changes", fontsize=16, pad=20)

savefig(fig, "05_engineering_summary.png")
plt.show()


## 7. Summary table

Notebook 05 exists to establish the replacement strategy for replication:

```text
one device
→ many optical frequency modes
→ many possible quantum channels
```


In [ ]:
summary = pd.DataFrame(
    [
        {
            "Concept": "Frequency multiplexing",
            "Engineering effect": "Many optical channels per device",
            "Why it matters": "Scaling shifts from hardware replication to optical modes.",
            "Next question": "How does a resonator generate many coherent frequencies?",
            "Next notebook": "07_quantum_frequency_combs",
        },
        {
            "Concept": "Frequency comb",
            "Engineering effect": "Evenly spaced resonances",
            "Why it matters": "Provides addressable optical resources.",
            "Next question": "How are mode ladders generated and stabilized?",
            "Next notebook": "07_quantum_frequency_combs",
        },
        {
            "Concept": "Integrated resonator",
            "Engineering effect": "One chip can replace many replicated devices",
            "Why it matters": "Packaging and alignment complexity can decrease.",
            "Next question": "How does Kerr nonlinearity create new modes?",
            "Next notebook": "11_kerr_nonlinearity",
        },
        {
            "Concept": "Mode resources",
            "Engineering effect": "Many distinguishable channels",
            "Why it matters": "Modes become the substrate for later squeezing and entanglement.",
            "Next question": "How do modes become quantum resources?",
            "Next notebook": "13_two_mode_squeezing",
        },
    ]
)

summary.to_csv(CSV / "05_summary.csv", index=False)
summary.to_json(JS / "05_summary.json", orient="records", indent=2)

summary


## 8. Export and download

This cell packages all outputs and starts a download.

It uses Colab's native downloader when available.  
For local Jupyter, it falls back to a clickable `FileLink`.


In [ ]:
zip_path = RES / "05_outputs.zip"

files_to_zip = [
    FIG / "05_replication_to_modes.png",
    FIG / "05_frequency_comb.png",
    FIG / "05_channel_scaling.png",
    FIG / "05_multiplexing_pipeline.png",
    FIG / "05_modes_as_resources.png",
    FIG / "05_engineering_summary.png",
    CSV / "05_summary.csv",
    JS / "05_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

summary


## Takeaways

- Frequency multiplexing shifts scaling from replicated devices to addressable optical modes.
- A resonator can provide a ladder of modes separated by the free spectral range.
- Modes are not yet entangled in this notebook; they are the resource axis.
- The next question is how these modes become a coherent quantum frequency comb.

## Next notebook

**07 — Quantum Frequency Combs**

```text
frequency modes
→ coherent comb structure
→ quantum harmonic oscillator modes
→ foundation for squeezing
```
